# APO Colab Bootstrap

This notebook sets up the environment for running ProTeGi prompt
optimization experiments on Colab Pro with data stored on Google Drive.

**Supported tasks:** FinanceBench, FinQA, FinDoc-RAG

**Prerequisites:**
- An `OPENAI_API_KEY` stored in Colab Secrets (key icon in the left sidebar).
- Vectorstores uploaded to Google Drive (see the layout section below).
- Optionally, Kaggle credentials in Colab Secrets (`KAGGLE_USERNAME` and `KAGGLE_KEY`) if using the BEM scorer.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Configuration

Edit `APO_ROOT` below to match where your project data lives on Drive.

The expected directory layout under `APO_ROOT` is:
```
data/
  FinanceBench/       # dataset_prepared.parquet + source jsonl files
  FinQa/              # dataset_prepared.parquet + train.json, dev.json, test.json
  Findoc/             # dataset_prepared.parquet + qa.json
references/
  FinanceBenchPdfs/   # downloaded automatically on first run
  FinDoc/             # markdown reference files
vectorstores/
  FinQa/finqa/<doc_id>/chroma.sqlite3   # one folder per document
  FinanceBench/financebench/<doc_id>/chroma.sqlite3
  FinDoc/findoc/<doc_id>/chroma.sqlite3
results/              # created automatically
prompts/              # seed prompt .txt files
```

In [ ]:
import os

APO_ROOT = "/content/drive/MyDrive/APO"

os.environ["APO_ROOT"] = APO_ROOT
os.environ["HF_HOME"] = os.path.join(APO_ROOT, ".cache", "hf")

os.makedirs(APO_ROOT, exist_ok=True)
for subdir in ["data", "references", "vectorstores", "results", "prompts"]:
    os.makedirs(os.path.join(APO_ROOT, subdir), exist_ok=True)

print(f"APO_ROOT = {APO_ROOT}")
print(f"HF_HOME  = {os.environ['HF_HOME']}")

## 3. Secrets

Store your `OPENAI_API_KEY` in **Colab Secrets** (the key icon in the
left sidebar). The cell below reads it from there so it never appears
in the notebook.

In [ ]:
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
assert os.environ["OPENAI_API_KEY"], "OPENAI_API_KEY not found in Colab Secrets."
print("OPENAI_API_KEY loaded.")

# Kaggle credentials for the BEM scorer.
try:
    kaggle_user = userdata.get("KAGGLE_USERNAME")
    kaggle_key = userdata.get("KAGGLE_KEY")
    if kaggle_user and kaggle_key:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        import json
        with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
            json.dump({"username": kaggle_user, "key": kaggle_key}, f)
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        print("Kaggle credentials written.")
    else:
        print("Kaggle credentials not set (BEM scorer will not be available).")
except Exception:
    print("Kaggle credentials not set (BEM scorer will not be available).")

## 4. Clone the repo and install dependencies

In [ ]:
REPO_DIR = "/content/APO"

if not os.path.isdir(REPO_DIR):
    !git clone -b colab-port https://github.com/CyprianBohojlo/APO.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
!pip install -q -r requirements-colab.txt

## 5. Verify vectorstore layout

The pipeline expects per-document Chroma stores at:
```
{APO_ROOT}/vectorstores/{TaskDir}/{task}/{doc_id}/chroma.sqlite3
```
For FinQA this is `vectorstores/FinQa/finqa/<doc_id>/chroma.sqlite3`.

Run the cell below to check whether your vectorstores are in the
expected location. If the count is 0 but you have uploaded them
to a different Drive path, either move them or create a symlink.

In [ ]:
import pathlib

TASK = "finqa"

task_vs_dirs = {
    "financebench": "FinanceBench/financebench",
    "finqa": "FinQa/finqa",
    "findoc": "FinDoc/findoc",
}

vs_path = pathlib.Path(APO_ROOT) / "vectorstores" / task_vs_dirs[TASK]
if vs_path.exists():
    docs = [d.name for d in vs_path.iterdir() if d.is_dir()]
    chroma_count = sum(1 for d in vs_path.iterdir() if (d / "chroma.sqlite3").exists())
    print(f"Found {len(docs)} doc folders, {chroma_count} with chroma.sqlite3")
    if docs:
        print(f"Sample: {docs[:5]}")
else:
    print(f"Directory does not exist: {vs_path}")
    print(f"Upload your vectorstores there, or symlink from wherever they live on Drive.")

## 6. Run the pipeline

Uncomment and edit the cells below as needed. Each step can be run
independently as long as its inputs already exist on Drive.

In [ ]:
# Prepare dataset
# !python prepare_data.py --task finqa --out_dir $APO_ROOT/data/FinQa

In [ ]:
# Build vectorstores
# !python vectorize.py --task finqa --data_dir $APO_ROOT/data/FinQa --vs_dir $APO_ROOT/vectorstores/FinQa

In [ ]:
# ProTeGi prompt optimization
# !python main.py \
#     --task finqa \
#     --data_dir $APO_ROOT/data/FinQa \
#     --prompts $APO_ROOT/prompts/seed.txt \
#     --rounds 6 \
#     --beam_size 4

In [ ]:
# Generate answers
# !python generate.py

In [ ]:
# Evaluate answers
# !python evaluate.py --metric gpt